In [1]:
# import the libraries
import pandas as pd
import json

In [2]:
df = pd.read_csv("../data/Milestone_2_contract_evaluation_dataset_nadvik-k-m.csv")
# Check required columns
required_cols = [
    "contract_text",
    "expected_apr",
    "expected_term",
    "expected_monthly_payment",
    "expected_penalty"
]
assert set(required_cols).issubset(df.columns), "Missing required columns"

df.head(10)


,contract_id,contract_text,expected_apr,expected_term,expected_monthly_payment,expected_penalty
0,1,This contract is entered between Prime Auto Fi...,9.75,36.0,825.0,late payment fee of $50
1,2,This leasing agreement between ABC Motors and ...,6.20,48.0,640.0,NaN
2,3,This agreement is signed between DriveEasy Lea...,NaN,24.0,510.0,early termination fee of $300
3,4,This auto loan contract between National Motor...,11.40,60.0,940.0,NaN
4,5,The contract between CarHub Leasing and Priya ...,NaN,36.0,720.0,late payment fee of $25
5,6,This loan agreement between AutoCredit Ltd and...,8.90,24.0,560.0,late payment penalty of $40
6,7,This financing contract is made between Veloci...,NaN,48.0,610.0,NaN
7,8,This agreement between SpeedDrive Finance and ...,10.50,36.0,NaN,NaN
8,9,This lease contract signed by Urban Auto and K...,NaN,18.0,495.0,early termination fee of $200
9,10,This auto loan document between Prime Wheels a...,7.80,60.0,710.0,NaN


In [3]:
df.columns

Index(['contract_id', 'contract_text', 'expected_apr', 'expected_term',
       'expected_monthly_payment', 'expected_penalty'],
      dtype='object')

In [4]:
expected_output = {
    "apr": None,
    "term_months": None,
    "monthly_payment": None,
    "penalty_clause": None
}


In [5]:
PROMPT_TEMPLATE = """
You are an information extraction assistant.

Extract ONLY the following fields from the contract text below:
- APR
- Term (in months)
- Monthly payment
- Penalty clause

Rules:
- Output must be valid JSON only
- If a value is NOT mentioned, return null
- Do NOT infer or guess values
- Do not add extra fields

Return exactly this JSON structure:
{{
  "apr": null,
  "term_months": null,
  "monthly_payment": null,
  "penalty_clause": null
}}

Contract text:
\"\"\"{contract_text}\"\"\"
"""


In [6]:
import json

def dummy_llm(prompt_text: str) -> str:
    
    return json.dumps({
        "apr": None,
        "term_months": None,
        "monthly_payment": None,
        "penalty_clause": None
    })

def extract_from_llm(text: str, llm_func):
    filled_prompt = PROMPT_TEMPLATE.format(contract_text=text)
    raw = llm_func(filled_prompt)
    return json.loads(raw)


In [7]:
sample_contracts = df.sample(5, random_state=42)

results = []
for _, row in sample_contracts.iterrows():
    extracted = extract_from_llm(row["contract_text"], dummy_llm)
    results.append({
        "contract_text": row["contract_text"],
        "llm_apr": extracted.get("apr"),
        "llm_term": extracted.get("term_months"),
        "llm_payment": extracted.get("monthly_payment"),
        "llm_penalty": extracted.get("penalty_clause"),
        "expected_apr": row["expected_apr"],
        "expected_term": row["expected_term"],
        "expected_payment": row["expected_monthly_payment"],
        "expected_penalty": row["expected_penalty"]
    })

results_df = pd.DataFrame(results)
print(results_df)


                                       contract_text llm_apr llm_term  \
0  This auto lease agreement between CarEase Ltd ...    None     None   
1  This auto financing agreement between National...    None     None   
2  This agreement between DriveSmart Leasing and ...    None     None   
3  This vehicle loan document between SpeedTrack ...    None     None   
4  This lease contract signed by Urban Auto and K...    None     None   

  llm_payment llm_penalty  expected_apr  expected_term  expected_payment  \
0        None        None           NaN           30.0             575.0   
1        None        None           NaN           72.0             890.0   
2        None        None           NaN           12.0             390.0   
3        None        None           NaN           48.0               NaN   
4        None        None           NaN           18.0             495.0   

                expected_penalty  
0                            NaN  
1                            NaN  

In [8]:
def compare_exact(val_llm, val_expected):
    if pd.isna(val_llm) and pd.isna(val_expected):
        return 1
    return 1 if val_llm == val_expected else 0

results_df["apr_match"] = results_df.apply(
    lambda r: compare_exact(r["llm_apr"], r["expected_apr"]), axis=1
)
results_df["term_match"] = results_df.apply(
    lambda r: compare_exact(r["llm_term"], r["expected_term"]), axis=1
)
results_df["payment_match"] = results_df.apply(
    lambda r: compare_exact(r["llm_payment"], r["expected_payment"]), axis=1
)
results_df["penalty_match"] = results_df.apply(
    lambda r: compare_exact(r["llm_penalty"], r["expected_penalty"]), axis=1
)


In [11]:
apr_accuracy = results_df["apr_match"].mean()*100
term_accuracy = results_df["term_match"].mean()*100
payment_accuracy = results_df["payment_match"].mean()*100
penalty_accuracy = results_df["penalty_match"].mean()*100

overall_accuracy = results_df[["apr_match","term_match","payment_match","penalty_match"]].values.mean()

accuracy_summary = pd.DataFrame({
    "Field": ["APR", "Term", "Payment", "Penalty", "Overall"],
    "Accuracy": [
        apr_accuracy,
        term_accuracy,
        payment_accuracy,
        penalty_accuracy,
        overall_accuracy
    ]
})

print("Extraction Results : ")
print(results_df)


Extraction Results : 
                                       contract_text llm_apr llm_term  \
0  This auto lease agreement between CarEase Ltd ...    None     None   
1  This auto financing agreement between National...    None     None   
2  This agreement between DriveSmart Leasing and ...    None     None   
3  This vehicle loan document between SpeedTrack ...    None     None   
4  This lease contract signed by Urban Auto and K...    None     None   

  llm_payment llm_penalty  expected_apr  expected_term  expected_payment  \
0        None        None           NaN           30.0             575.0   
1        None        None           NaN           72.0             890.0   
2        None        None           NaN           12.0             390.0   
3        None        None           NaN           48.0               NaN   
4        None        None           NaN           18.0             495.0   

                expected_penalty  apr_match  term_match  payment_match  \
0       

In [12]:
print("\n Accuracy Summary :" )
print(accuracy_summary)


 Accuracy Summary :
     Field  Accuracy
0      APR    100.00
1     Term      0.00
2  Payment     20.00
3  Penalty     60.00
4  Overall      0.45
